# S3 : tester le modèle sur un fichier audio

Donnez un fichier audio en arabe (wav, flac, mp3…, n'importe quelle fréquence : il est ramené en mono 16 kHz). Le notebook le découpe en morceaux de 20 s au plus, aux moments les plus calmes, puis transcrit chaque morceau avec la chaîne S3 :

encodeur Cohere Transcribe Arabic (gelé) → **projecteur entraîné** → Qwen2.5-1.5B-Instruct (gelé).

- `CHEMIN_AUDIO = None` : on prend 60 s d'une émission du **test** MGB-2 (jamais vue à l'entraînement), avec sa transcription de référence pour comparer.
- Le modèle n'a appris que la **transcription arabe** (temps 1). La **traduction française** est faite ensuite par le même Qwen2.5-1.5B-Instruct, en mode texte (« Traduis en français… »), sans réentraînement : c'est une cascade. Le temps 2 apprendra la traduction directe depuis l'audio.
- Toute la logique est dans `code/projecteur/transcrire_s3.py` ; réglages dans `configs/s3.yaml` (sections `transcription` et `temps_1_50k`).

**Ne pas committer ce notebook avec ses sorties** si l'audio vient de MGB-2 (licence de recherche QCRI).

In [20]:
import importlib
import transcrire_s3
importlib.reload(transcrire_s3)
transcripteur.__class__ = transcrire_s3.TranscripteurS3  # garde l'encodeur et le LLM déjà chargés

In [17]:
# À modifier : chemin de votre fichier audio, ou None pour un extrait du test MGB-2
CHEMIN_AUDIO = None
DUREE_EXTRAIT_S = 60          # durée de l'extrait MGB-2 quand CHEMIN_AUDIO est None
SECTION = "temps_1_50k"        # entraînement dont on prend le meilleur projecteur
FICHIER_SRT = None             # par exemple "sous_titres_ar.srt" : sous-titres arabes
FICHIER_SRT_FR = None          # par exemple "sous_titres_fr.srt" : sous-titres français

In [18]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import yaml

racine = Path.cwd()
while not (racine / "configs" / "s3.yaml").exists() and racine.parent != racine:
    racine = racine.parent
for dossier in ("projecteur", "ecoute", "donnees", "traduction", "mesures", "demo"):
    if str(racine / "code" / dossier) not in sys.path:
        sys.path.insert(0, str(racine / "code" / dossier))

from transcrire_s3 import TranscripteurS3, charger_audio, transcripteur_factice

ESSAI = os.environ.get("S3_NOTEBOOK_ESSAI") == "1"   # petits modèles factices (tests), sans GPU
depart = time.perf_counter()
transcripteur = transcripteur_factice(SECTION) if ESSAI else TranscripteurS3.charger(SECTION)
print(f"modèles chargés en {time.perf_counter() - depart:.0f} s")

Loading weights: 100%|██████████| 338/338 [00:00<00:00, 8390.00it/s]


modèles chargés en 23 s


## Choisir l'audio

In [10]:
references = []
if ESSAI:
    signal = (0.1 * np.random.default_rng(0).standard_normal(30 * 16000)).astype(np.float32)
    origine = "bruit factice (mode essai)"
elif CHEMIN_AUDIO is not None:
    signal = charger_audio(CHEMIN_AUDIO)
    origine = str(CHEMIN_AUDIO)
else:
    from benchmark_traduction import lire_xml_mgb2
    config_encodage = yaml.safe_load((racine / "configs" / "encodage_s3.yaml").read_text(encoding="utf-8"))
    sources = config_encodage["sources"]
    racine_mgb2 = Path(sources["racine_mgb2"]) if Path(sources["racine_mgb2"]).is_absolute() else racine / sources["racine_mgb2"]
    with open(racine / config_encodage["sortie"]["dossier"] / "test.jsonl", encoding="utf-8") as fichier:
        premier = json.loads(fichier.readline())
    # Extrait calé sur des segments entiers : du début d'un segment à la fin du dernier segment
    # qui tient dans DUREE_EXTRAIT_S, pour que l'audio et la référence couvrent la même parole.
    segments_xml = sorted(lire_xml_mgb2(racine_mgb2 / sources["sous_dossier_xml"] / f"{premier['episode']}.xml"),
                          key=lambda s: s["debut"])
    suite = [s for s in segments_xml if s["debut"] >= premier["debut"]]
    references = [s for s in suite if s["fin"] <= suite[0]["debut"] + DUREE_EXTRAIT_S] or suite[:1]
    debut, fin = references[0]["debut"], references[-1]["fin"]
    signal = charger_audio(racine_mgb2 / premier["wav"])[int(debut * 16000): int(fin * 16000)]
    origine = f"MGB-2 test, émission {premier['episode']}, {debut:.1f} à {fin:.1f} s"
print(origine, f": {len(signal) / 16000:.1f} s")
if not ESSAI:
    from IPython.display import Audio, display
    display(Audio(signal, rate=16000))

MGB-2 test, émission 1E084D4F-5305-46E1-8AB2-7EBB9DCA5F2E, 1082.3 à 1115.3 s : 33.0 s


## Transcrire

In [11]:
depart = time.perf_counter()
segments = transcripteur.transcrire_signal(signal)
duree_calcul = time.perf_counter() - depart
for s in segments:
    print(f"[{s['debut_s']:6.1f} - {s['fin_s']:6.1f} s] {s['texte']}")
print(f"\n{len(signal) / 16000:.1f} s d'audio transcrites en {duree_calcul:.1f} s "
      f"(RTF {duree_calcul / (len(signal) / 16000):.3f}, encodage compris)")
if FICHIER_SRT:
    from srt import sauvegarder_srt
    print("sous-titres :", sauvegarder_srt(segments, FICHIER_SRT))

[   0.0 -   17.9 s] كلما زادت الزيادة زاد عملنا والزيادة تجبرنا على قيادة الشاحنة بعناية أكثر لأن الزيادة مكلفة وأحياناً تصل إلى ملايين الروبية . فتطورات التكنولوجيا أدخلت عمليات اللصق التي تعد أقل جودةً وتكلفةً من الرسم باليد ،
[  17.9 -   33.0 s] ولم يعد تزيين الشاحنات في ضوء ذلك يتطلب الكثير من الإبداع والهدف في النهاية جعل الحافلة جميلة ، معارض بيع الزينة ازدهرت واتباع هناك فنانون مختصون في شؤون عمل اللواسق

33.0 s d'audio transcrites en 23.4 s (RTF 0.710, encodage compris)


## Comparer à la référence (extrait MGB-2 seulement)

Référence : les segments du XML MGB-2 de l'extrait (calé sur des segments entiers), mis bout à bout. Les morceaux du système et les segments de référence n'ont pas les mêmes bornes : on compare donc les deux textes complets. Les blancs entre segments de référence peuvent contenir de la parole non transcrite dans MGB-2, ce qui ajoute quelques erreurs.

In [12]:
if references:
    from scores import calculer_wer_cer
    texte_reference = " ".join(s["ar"] for s in references)
    texte_systeme = " ".join(s["texte"] for s in segments)
    print("RÉFÉRENCE :", texte_reference, "\n")
    print("S3        :", texte_systeme, "\n")
    scores = calculer_wer_cer([texte_reference], [texte_systeme])
    print(f"WER {scores['wer_pct']} %, CER {scores['cer_pct']} % sur l'extrait ({len(references)} segments de référence)")
else:
    print("pas de référence : audio fourni par l'utilisateur ou mode essai")

RÉFÉRENCE : كلما زادت الزينة زاد عملنا والزينة تجبرنا على قيادة الشاحنة بعنايةٍ أكثر لأنّ الزينة مكلفة وأحياناً تصل إلى ملايين الروبيات . فثورة التكنولوجيا أدخلت عمليات اللصق التي تُعد أقل جودةً وتكلفةً من الرسم باليد ، ولم يعد تزيين الشاحنات في ضوء ذلك تطلب الكثير من الإبداع والمهم في النهاية جعل الحافلة جميلة ، محال بيع الزينة ازدهرت وبات هناك فنانون مختصون في شؤون عمل اللواصق 

S3        : كلما زادت الزيادة زاد عملنا والزيادة تجبرنا على قيادة الشاحنة بعناية أكثر لأن الزيادة مكلفة وأحياناً تصل إلى ملايين الروبية . فتطورات التكنولوجيا أدخلت عمليات اللصق التي تعد أقل جودةً وتكلفةً من الرسم باليد ، ولم يعد تزيين الشاحنات في ضوء ذلك يتطلب الكثير من الإبداع والهدف في النهاية جعل الحافلة جميلة ، معارض بيع الزينة ازدهرت واتباع هناك فنانون مختصون في شؤون عمل اللواسق 

WER 16.13 %, CER 6.44 % sur l'extrait (5 segments de référence)


## Traduire en français

Chaque morceau transcrit est traduit seul par Qwen2.5-1.5B-Instruct, avec un message système de traducteur et la consigne « Traduis ce texte en français. » (section `traduction_texte` de `configs/s3.yaml`). Une introduction du type « Voici la traduction : » est retirée ; si de l'arabe reste dans la sortie, le texte est retraduit avec une seconde consigne.

Pour un extrait MGB-2, on traduit aussi la **référence arabe** : l'écart entre les deux traductions montre ce que coûtent les erreurs de transcription. MGB-2 n'a pas de traduction française de référence, donc pas de score ici ; la qualité d'un modèle de 1,5B en traduction arabe → français est à juger à la lecture.

In [14]:
depart = time.perf_counter()
traductions = transcripteur.traduire([s["texte"] for s in segments])
print(f"traduit en {time.perf_counter() - depart:.1f} s\n")
for s, fr in zip(segments, traductions):
    print(f"[{s['debut_s']:6.1f} - {s['fin_s']:6.1f} s]")
    print("  AR :", s["texte"])
    print("  FR :", fr)
if FICHIER_SRT_FR:
    from srt import sauvegarder_srt
    print("sous-titres français :", sauvegarder_srt([dict(s, texte=fr) for s, fr in zip(segments, traductions)], FICHIER_SRT_FR))

traduit en 3.0 s

[   0.0 -   17.9 s]
  AR : كلما زادت الزيادة زاد عملنا والزيادة تجبرنا على قيادة الشاحنة بعناية أكثر لأن الزيادة مكلفة وأحياناً تصل إلى ملايين الروبية . فتطورات التكنولوجيا أدخلت عمليات اللصق التي تعد أقل جودةً وتكلفةً من الرسم باليد ،
  FR : Tant que la quantité augmente, nous travaillons plus et la quantité nous oblige à conduire la camionnette avec plus de prudence, car la quantité est coûteuse et parfois atteint des millions de rupees. Les avancées technologiques ont introduit des opérations de coller qui sont moins de qualité et moins coûteuses que les dessins manuels.
[  17.9 -   33.0 s]
  AR : ولم يعد تزيين الشاحنات في ضوء ذلك يتطلب الكثير من الإبداع والهدف في النهاية جعل الحافلة جميلة ، معارض بيع الزينة ازدهرت واتباع هناك فنانون مختصون في شؤون عمل اللواسق
  FR : Et malgré cela, le décor des camions ne demande plus beaucoup d'innovation. Le but final est de rendre la voiture belle, et les marchés de vente de décoration ont connu un regain d'intérêt. Des artiste

In [15]:
if references:
    traduction_systeme = " ".join(traductions)
    traduction_reference = " ".join(transcripteur.traduire([s["ar"] for s in references]))
    print("FR depuis la transcription S3  :", traduction_systeme, "\n")
    print("FR depuis la référence arabe   :", traduction_reference)
else:
    print("pas de référence : audio fourni par l'utilisateur ou mode essai")

FR depuis la transcription S3  : Tant que la quantité augmente, nous travaillons plus et la quantité nous oblige à conduire la camionnette avec plus de prudence, car la quantité est coûteuse et parfois atteint des millions de rupees. Les avancées technologiques ont introduit des opérations de coller qui sont moins de qualité et moins coûteuses que les dessins manuels. Et malgré cela, le décor des camions ne demande plus beaucoup d'innovation. Le but final est de rendre la voiture belle, et les marchés de vente de décoration ont connu un regain d'intérêt. Des artistes spécialisés dans l'art de la tapisserie ont également pris de l'ampleur. 

FR depuis la référence arabe   : Plus la décoration est grande, plus nous travaillons. La décoration nous oblige à conduire la camionnette avec plus de prudence. Car la bijouterie est coûteuse et parfois coûteuse à des millions de roupies. La révolution technologique a introduit des opérations de coller. cela qui est la moins de qualité et coûte le 

## Traduire directement depuis l'audio (sans transcription)

Ici le texte arabe n'existe jamais : les vecteurs audio sortis du **projecteur** sont donnés tels quels au LLM, suivis de la consigne « Traduis en français. » (`consignes.temps_2_traduction` de `configs/s3.yaml`) :

`[système][audio projeté]["Traduis en français."]` → Qwen2.5-1.5B-Instruct

Le projecteur n'a appris que la transcription (temps 1) : c'est un essai **sans réentraînement**. Si le LLM recopie l'arabe au lieu de traduire, c'est que le projecteur a appris des vecteurs liés à la consigne de transcription ; le temps 2 (LoRA + projecteur) est là pour apprendre cette tâche. La part de caractères arabes dans la sortie mesure cet effet.

Pour essayer une autre consigne : `transcripteur.modele.consignes["traduction"] = "..."` avant de relancer la cellule.


In [21]:
import re

depart = time.perf_counter()
segments_directs = transcripteur.traduire_signal(signal)
print(f"traduit directement en {time.perf_counter() - depart:.1f} s "
      f"(consigne : {transcripteur.modele.consignes['traduction']!r})\n")

def part_arabe(texte):
    lettres = [c for c in texte if c.isalpha()]
    return sum(bool(re.match(r"[\u0600-\u06FF]", c)) for c in lettres) / max(1, len(lettres))

for s, fr_cascade in zip(segments_directs, traductions):
    print(f"[{s['debut_s']:6.1f} - {s['fin_s']:6.1f} s]  (arabe dans la sortie directe : {part_arabe(s['texte']):.0%})")
    print("  DIRECT  :", s["texte"])
    print("  CASCADE :", fr_cascade)
texte_direct = " ".join(s["texte"] for s in segments_directs)
print(f"\npart de caractères arabes : directe {part_arabe(texte_direct):.0%}, cascade {part_arabe(' '.join(traductions)):.0%}")


traduit directement en 9.8 s (consigne : 'Traduis en français.')

[   0.0 -   17.9 s]  (arabe dans la sortie directe : 100%)
  DIRECT  : كلما زادت الزيادة زاد عملنا والزيادة تجبرنا على قيادة الشاحنة بعناية أكثر لأن الزيادة مكلفة وأحيانا تصل إلى ملايين الروبية . فتطورات التكنولوجيا أدخلت عمليات اللصق التي تعد أقل جودة وتكلفة من الرسم باليد ،
  CASCADE : Tant que la quantité augmente, nous travaillons plus et la quantité nous oblige à conduire la camionnette avec plus de prudence, car la quantité est coûteuse et parfois atteint des millions de rupees. Les avancées technologiques ont introduit des opérations de coller qui sont moins de qualité et moins coûteuses que les dessins manuels.
[  17.9 -   33.0 s]  (arabe dans la sortie directe : 100%)
  DIRECT  : ولم يعد تزيين الشاحنات في ضوء ذلك يتطلب الكثير من الإبداع والهدف في النهاية جعل الحافلة جميلة ، معارض بيع الزينة ازدهرت واتباع هناك فنانون مختصون في شؤون عمل اللواسق
  CASCADE : Et malgré cela, le décor des camions ne demande plus beauc